In [1]:
"""Phase D Benchmark: compacted vs predicated vs dense attention.

Paste this ENTIRE cell into a Kaggle TPU notebook. Zero imports from the repo.
Uses standard JAX ops only — no Pallas, no BlockSpec, no Mosaic.

This measures the ACTUAL Δτ from stream compaction: fewer blocks gathered
and matmul'd means fewer FLOPs means less wall-clock time.
"""

import jax
import jax.numpy as jnp
import time

BS = 512; HD = 128; NH = 4; WARMUP = 10; REPS = 30

# ============================================================================
# Data generation
# ============================================================================
def make_data(sl):
    k1,k2,k3 = jax.random.split(jax.random.PRNGKey(0), 3)
    return (jax.random.normal(k1, (1,NH,HD), dtype=jnp.bfloat16),
            jax.random.normal(k2, (sl,NH,HD), dtype=jnp.bfloat16),
            jax.random.normal(k3, (sl,NH,HD), dtype=jnp.bfloat16))

def make_mask(nb, pct):
    if pct == 0: return jnp.ones((nb,NH), dtype=jnp.bool_)
    n = int(nb * pct / 100)
    m = jnp.ones(nb, dtype=jnp.bool_).at[:n].set(False)
    return jnp.broadcast_to(
        m[jax.random.permutation(jax.random.PRNGKey(42), nb)][:, None],
        (nb, NH))

def bench(label, fn):
    for _ in range(WARMUP): fn().block_until_ready()
    ts = []
    for _ in range(REPS):
        t0 = time.perf_counter(); fn().block_until_ready()
        ts.append((time.perf_counter() - t0) * 1000)
    ts.sort(); med = ts[len(ts)//2]
    print(f"  {label}: {med:.3f} ms")
    return med

# ============================================================================
# Stream compaction (pure JAX — emulates the HLO pass prefix sum)
# ============================================================================
def stream_compact(mask):
    if mask.ndim == 2: mask_1d = jnp.any(mask, axis=1)
    else: mask_1d = mask
    nb = mask_1d.shape[0]
    iota = jnp.arange(nb, dtype=jnp.int32)
    keys = jnp.where(mask_1d, iota, nb + iota)
    return jnp.argsort(keys, stable=True), jnp.sum(mask_1d).astype(jnp.int32)

# ============================================================================
# Three attention implementations
# ============================================================================

@jax.jit
def dense_attn(q, k, v):
    """Baseline: full dense attention, no masking."""
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k.astype(jnp.float32)) / sc
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, v.astype(jnp.float32)).astype(jnp.bfloat16)

@jax.jit
def predicated_attn(q, k, v, mask):
    """v1: compute ALL blocks, mask logits. MXU still fires for evicted blocks."""
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k.astype(jnp.float32)) / sc
    token_mask = jnp.repeat(mask, BS, axis=0)  # (seq_k, NH)
    lo = jnp.where(token_mask[None,:,:], lo, -1e9)
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, v.astype(jnp.float32)).astype(jnp.bfloat16)

@jax.jit
def compacted_attn(q, k, v, mask):
    """v2: stream compact → gather active blocks → dense attention on K blocks only."""
    nb = k.shape[0] // BS
    active_idx, num_active = stream_compact(mask)

    # Gather active blocks
    k_blocks = k.reshape(nb, BS, NH, HD)
    v_blocks = v.reshape(nb, BS, NH, HD)
    k_comp = k_blocks[active_idx]  # (nb, BS, NH, HD) — first K are valid
    v_comp = v_blocks[active_idx]
    k_flat = k_comp.reshape(nb * BS, NH, HD)
    v_flat = v_comp.reshape(nb * BS, NH, HD)

    # Validity mask: only first num_active * BS tokens are real
    valid = jnp.arange(nb * BS) < (num_active * BS)

    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k_flat.astype(jnp.float32)) / sc
    lo = jnp.where(valid[None,:,None], lo, -1e9)
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, v_flat.astype(jnp.float32)).astype(jnp.bfloat16)

# ============================================================================
# Run benchmarks
# ============================================================================
print(f"Platform: {jax.devices()[0].device_kind}, JAX {jax.__version__}")
print(f"Config: Q=1, heads={NH}, d={HD}, block={BS}, warmup={WARMUP}, reps={REPS}")

results = {}
for seq in [8192, 16384, 32768, 65536]:
    nb = seq // BS
    q, k, v = make_data(seq)
    print(f"\n{'='*70}")
    print(f"  {seq} tokens ({nb} blocks)")
    print(f"{'='*70}")

    t_dense = bench("dense_0%", lambda: dense_attn(q, k, v))

    for pct in [0, 25, 50, 75, 90]:
        m = make_mask(nb, pct)
        t_pred = bench(f"predicated_{pct}%", lambda m=m: predicated_attn(q, k, v, m))
        t_comp = bench(f"compacted_{pct}%",  lambda m=m: compacted_attn(q, k, v, m))
        results[(seq, pct)] = {'dense': t_dense, 'pred': t_pred, 'comp': t_comp}

# Summary
print(f"\n{'='*70}")
print(f"  SUMMARY: Δτ = (predicated - compacted) / predicated")
print(f"{'='*70}")
print(f"{'seq':>6} | {'evict%':>6} | {'pred ms':>8} | {'comp ms':>8} | {'Δτ':>8} | {'speedup':>8}")
print("-" * 60)
for (seq, pct), r in sorted(results.items()):
    dt = (r['pred'] - r['comp']) / r['pred'] * 100
    spd = r['pred'] / r['comp'] if r['comp'] > 0 else 0
    print(f"{seq:>6} | {pct:>6} | {r['pred']:>8.3f} | {r['comp']:>8.3f} | {dt:>7.1f}% | {spd:>7.2f}x")


/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1780418857.473981      14 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


Platform: TPU v5 lite, JAX 0.10.1
Config: Q=1, heads=4, d=128, block=512, warmup=10, reps=30

  8192 tokens (16 blocks)
  dense_0%: 0.252 ms
  predicated_0%: 0.254 ms
  compacted_0%: 0.398 ms
  predicated_25%: 0.246 ms
  compacted_25%: 0.392 ms
  predicated_50%: 0.245 ms
  compacted_50%: 0.389 ms
  predicated_75%: 0.244 ms
  compacted_75%: 0.394 ms
  predicated_90%: 0.246 ms
  compacted_90%: 0.391 ms

  16384 tokens (32 blocks)
  dense_0%: 0.433 ms
  predicated_0%: 0.431 ms
  compacted_0%: 0.547 ms
  predicated_25%: 0.428 ms
  compacted_25%: 0.523 ms
  predicated_50%: 0.397 ms
  compacted_50%: 0.504 ms
  predicated_75%: 0.397 ms
  compacted_75%: 0.491 ms
  predicated_90%: 0.395 ms
  compacted_90%: 0.496 ms

  32768 tokens (64 blocks)
  dense_0%: 0.395 ms
  predicated_0%: 0.418 ms
  compacted_0%: 0.604 ms
  predicated_25%: 0.411 ms
  compacted_25%: 0.611 ms
  predicated_50%: 0.402 ms
  compacted_50%: 0.604 ms
  predicated_75%: 0.399 ms
  compacted_75%: 0.605 ms
  predicated_90%: 0.419 m

In [2]:
import jax, jax.numpy as jnp, time
from functools import partial

BS=512; HD=128; NH=4; WARMUP=10; REPS=30

def make_data(sl):
    k1,k2,k3=jax.random.split(jax.random.PRNGKey(0),3)
    return (jax.random.normal(k1,(1,NH,HD),dtype=jnp.bfloat16),
            jax.random.normal(k2,(sl,NH,HD),dtype=jnp.bfloat16),
            jax.random.normal(k3,(sl,NH,HD),dtype=jnp.bfloat16))

def make_mask(nb,pct):
    if pct==0: return jnp.ones((nb,NH),dtype=jnp.bool_)
    n=int(nb*pct/100); m=jnp.ones(nb,dtype=jnp.bool_).at[:n].set(False)
    return jnp.broadcast_to(m[jax.random.permutation(jax.random.PRNGKey(42),nb)][:,None],(nb,NH))

def bench(label,fn):
    for _ in range(WARMUP): fn().block_until_ready()
    ts=[]
    for _ in range(REPS):
        t0=time.perf_counter(); fn().block_until_ready(); ts.append((time.perf_counter()-t0)*1000)
    ts.sort(); med=ts[len(ts)//2]; print(f"  {label}: {med:.3f} ms"); return med

def stream_compact(mask):
    if mask.ndim==2: mask_1d=jnp.any(mask,axis=1)
    else: mask_1d=mask
    nb=mask_1d.shape[0]
    iota=jnp.arange(nb,dtype=jnp.int32)
    keys=jnp.where(mask_1d,iota,nb+iota)
    return jnp.argsort(keys,stable=True), jnp.sum(mask_1d).astype(jnp.int32)

# Predicated: full matmul, mask logits
@jax.jit
def predicated(q,k,v,mask):
    sc=jnp.sqrt(jnp.float32(HD))
    lo=jnp.einsum('qhd,khd->qkh',q.astype(jnp.float32),k.astype(jnp.float32))/sc
    tm=jnp.repeat(mask,BS,axis=0)
    lo=jnp.where(tm[None,:,:],lo,-1e9)
    w=jax.nn.softmax(lo,axis=1)
    return jnp.einsum('qkh,khd->qhd',w,v.astype(jnp.float32)).astype(jnp.bfloat16)

# Bucketed: gather K blocks → matmul on (bucket*BS) tokens, NOT (M*BS)
@partial(jax.jit, static_argnums=(4,))
def bucketed(q, k_blocks, v_blocks, active_idx, bucket):
    """Attention on exactly `bucket` blocks. Bucket is STATIC → smaller matmul."""
    idx = active_idx[:bucket]
    ka = k_blocks[idx].reshape(bucket*BS, NH, HD)  # SMALLER tensor
    va = v_blocks[idx].reshape(bucket*BS, NH, HD)
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), ka.astype(jnp.float32)) / sc
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, va.astype(jnp.float32)).astype(jnp.bfloat16)

def next_bucket(k, max_blocks):
    """Round up to nearest power of 2, capped at max_blocks."""
    if k <= 0: return 1
    b = 1
    while b < k: b *= 2
    return min(b, max_blocks)

print(f"Platform: {jax.devices()[0].device_kind}, JAX {jax.__version__}")
print(f"Config: Q=1, heads={NH}, d={HD}, block={BS}")

for seq in [8192, 16384, 32768, 65536]:
    nb = seq // BS
    q, k, v = make_data(seq)
    k_blocks = k.reshape(nb, BS, NH, HD)
    v_blocks = v.reshape(nb, BS, NH, HD)

    print(f"\n{'='*70}")
    print(f"  {seq} tokens ({nb} blocks)")
    print(f"{'='*70}")

    for pct in [0, 50, 75, 90]:
        m = make_mask(nb, pct)
        t_pred = bench(f"pred_{pct}%", lambda m=m: predicated(q,k,v,m))

        aidx, nact = stream_compact(m)
        nact_val = int(nact)
        bkt = next_bucket(nact_val, nb)

        # Warmup the bucket (first call compiles)
        for _ in range(3): bucketed(q, k_blocks, v_blocks, aidx, bkt).block_until_ready()

        t_bkt = bench(f"bucket_{pct}% (K={nact_val},bkt={bkt})",
                      lambda ai=aidx, b=bkt: bucketed(q, k_blocks, v_blocks, ai, b))

        dt = (t_pred - t_bkt) / t_pred * 100
        print(f"    -> Δτ = {dt:+.1f}%  ({t_pred/t_bkt:.2f}x)")

Platform: TPU v5 lite, JAX 0.10.1
Config: Q=1, heads=4, d=128, block=512

  8192 tokens (16 blocks)
  pred_0%: 0.228 ms
  bucket_0% (K=16,bkt=16): 0.333 ms
    -> Δτ = -45.8%  (0.69x)
  pred_50%: 0.202 ms
  bucket_50% (K=8,bkt=8): 0.204 ms
    -> Δτ = -0.6%  (0.99x)
  pred_75%: 0.190 ms
  bucket_75% (K=4,bkt=4): 0.185 ms
    -> Δτ = +2.2%  (1.02x)
  pred_90%: 0.231 ms
  bucket_90% (K=2,bkt=2): 0.148 ms
    -> Δτ = +36.0%  (1.56x)

  16384 tokens (32 blocks)
  pred_0%: 0.383 ms
  bucket_0% (K=32,bkt=32): 0.484 ms
    -> Δτ = -26.4%  (0.79x)
  pred_50%: 0.390 ms
  bucket_50% (K=16,bkt=16): 0.345 ms
    -> Δτ = +11.4%  (1.13x)
  pred_75%: 0.383 ms
  bucket_75% (K=8,bkt=8): 0.196 ms
    -> Δτ = +48.8%  (1.95x)
  pred_90%: 0.391 ms
  bucket_90% (K=4,bkt=4): 0.166 ms
    -> Δτ = +57.4%  (2.35x)

  32768 tokens (64 blocks)
  pred_0%: 0.368 ms
  bucket_0% (K=64,bkt=64): 0.558 ms
    -> Δτ = -51.8%  (0.66x)
  pred_50%: 0.369 ms
  bucket_50% (K=32,bkt=32): 0.466 ms
    -> Δτ = -26.3%  (0.79x)
  

In [5]:
"""Phase D.5: XLA Loop Indirection Benchmark — fori_loop + dynamic_slice.
NO Pallas. NO intermediate buffer. NO libtpu version dependency.
DMA engine fetches directly from original KV-cache via computed address.
Loop runs exactly num_active iterations (dynamic bound, one JIT).
Paste this entire cell into a Kaggle TPU v5 notebook.
"""
import jax
import jax.numpy as jnp
from functools import partial
import time
BS = 512; HD = 128; NH = 4; WARMUP = 10; REPS = 30
# ============================================================
# Data & utilities (identical to previous benchmarks)
# ============================================================
def make_data(sl):
    k1, k2, k3 = jax.random.split(jax.random.PRNGKey(0), 3)
    return (
        jax.random.normal(k1, (1, NH, HD), dtype=jnp.bfloat16),
        jax.random.normal(k2, (sl, NH, HD), dtype=jnp.bfloat16),
        jax.random.normal(k3, (sl, NH, HD), dtype=jnp.bfloat16),
    )
def make_mask(nb, pct):
    if pct == 0: return jnp.ones(nb, dtype=jnp.bool_)
    n = int(nb * pct / 100)
    m = jnp.ones(nb, dtype=jnp.bool_).at[:n].set(False)
    return m[jax.random.permutation(jax.random.PRNGKey(42), nb)]
def stream_compact(mask):
    nb = mask.shape[0]
    iota = jnp.arange(nb, dtype=jnp.int32)
    keys = jnp.where(mask, iota, nb + iota)
    return jnp.argsort(keys, stable=True), jnp.sum(mask).astype(jnp.int32)
def bench(label, fn):
    for _ in range(WARMUP): fn().block_until_ready()
    ts = []
    for _ in range(REPS):
        t0 = time.perf_counter(); fn().block_until_ready()
        ts.append((time.perf_counter() - t0) * 1000)
    ts.sort(); med = ts[len(ts)//2]
    print(f"  {label}: {med:.3f} ms")
    return med
# ============================================================
# XLA loop indirection: fori_loop + dynamic_slice
# ============================================================
@jax.jit
def indirect_loop_attn(q, k_cache, v_cache, active_indices, num_active):
    """OrthoCache attention via native XLA loop indirection.
    - fori_loop compiles to a hardware loop on TPU
    - dynamic_slice compiles to DMA with computed address
    - No intermediate buffer, no gather, no Pallas
    - num_active is a TRACED int — one JIT handles all eviction rates
    """
    seq_q, num_heads, head_dim = q.shape
    scale = jnp.sqrt(jnp.float32(head_dim))
    # Online softmax accumulators
    m_init = jnp.full((seq_q, num_heads), -1e30, dtype=jnp.float32)
    l_init = jnp.zeros((seq_q, num_heads), dtype=jnp.float32)
    o_init = jnp.zeros((seq_q, num_heads, head_dim), dtype=jnp.float32)
    def body(i, carry):
        m_prev, l_prev, o_prev = carry
        # Scalar register read: which block to fetch
        real_idx = active_indices[i]
        # ONE HBM TOUCH: DMA fetches directly from original cache
        k_block = jax.lax.dynamic_slice(
            k_cache,
            (real_idx * BS, 0, 0),
            (BS, num_heads, head_dim)
        )
        v_block = jax.lax.dynamic_slice(
            v_cache,
            (real_idx * BS, 0, 0),
            (BS, num_heads, head_dim)
        )
        # Logits: (seq_q, BS, NH)
        logits = jnp.einsum('qhd,khd->qkh',
                            q.astype(jnp.float32),
                            k_block.astype(jnp.float32)) / scale
        # Online softmax update
        m_block = jnp.max(logits, axis=1)              # (seq_q, NH)
        m_next = jnp.maximum(m_prev, m_block)
        exp_logits = jnp.exp(logits - m_next[:, None, :])
        exp_prev = jnp.exp(m_prev - m_next)
        l_block = jnp.sum(exp_logits, axis=1)           # (seq_q, NH)
        l_next = l_prev * exp_prev + l_block
        # Weighted V: (seq_q, NH, HD)
        v_agg = jnp.einsum('qkh,khd->qhd',
                           exp_logits,
                           v_block.astype(jnp.float32))
        o_next = o_prev * exp_prev[:, :, None] + v_agg
        return m_next, l_next, o_next
    m_f, l_f, o_f = jax.lax.fori_loop(0, num_active, body, (m_init, l_init, o_init))
    return (o_f / l_f[:, :, None]).astype(jnp.bfloat16)
# ============================================================
# Baselines (identical to previous benchmarks)
# ============================================================
@jax.jit
def dense_attn(q, k, v):
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k.astype(jnp.float32)) / sc
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, v.astype(jnp.float32)).astype(jnp.bfloat16)
@jax.jit
def predicated_attn(q, k, v, mask):
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k.astype(jnp.float32)) / sc
    lo = jnp.where(jnp.repeat(mask, BS)[None, :, None], lo, -1e9)
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, v.astype(jnp.float32)).astype(jnp.bfloat16)
# ============================================================
# Correctness check
# ============================================================
print(f"Platform: {jax.devices()[0].device_kind}, JAX {jax.__version__}")
print(f"Config: Q=1, heads={NH}, d={HD}, block={BS}")
print(f"Method: jax.lax.fori_loop + jax.lax.dynamic_slice (native XLA)\n")
print("=== Correctness check (16K, 50% eviction) ===")
q_c, k_c, v_c = make_data(16384)
m_c = make_mask(32, 50)
aidx_c, nact_c = stream_compact(m_c)
out_pred  = predicated_attn(q_c, k_c, v_c, m_c)
out_loop  = indirect_loop_attn(q_c, k_c, v_c, aidx_c, nact_c)
err = jnp.max(jnp.abs(out_pred.astype(jnp.float32) - out_loop.astype(jnp.float32)))
print(f"  predicated vs loop-indirect max err: {err:.6f}")
assert err < 0.05, f"Output mismatch: {err}"
print("  PASSED\n")
# ============================================================
# Benchmark
# ============================================================
results = {}
for seq in [8192, 16384, 32768, 65536]:
    nb = seq // BS
    q, k, v = make_data(seq)
    print(f"{'='*70}")
    print(f"  {seq} tokens ({nb} blocks)")
    print(f"{'='*70}")
    t_dense = bench("dense", lambda: dense_attn(q, k, v))
    for pct in [0, 50, 75, 90]:
        m = make_mask(nb, pct)
        t_pred = bench(f"pred_{pct}%", lambda m=m: predicated_attn(q, k, v, m))
        aidx, nact = stream_compact(m)
        t_loop = bench(f"loop_{pct}% (K={int(nact)})",
                      lambda ai=aidx, na=nact: indirect_loop_attn(q, k, v, ai, na))
        dt = (t_pred - t_loop) / t_pred * 100
        spd = t_pred / t_loop if t_loop > 0 else 0
        print(f"    -> Δτ = {dt:+.1f}%  ({spd:.2f}x)")
        results[(seq, pct)] = {
            'dense': t_dense, 'pred': t_pred,
            'loop': t_loop, 'K': int(nact)
        }
# ============================================================
# Summary
# ============================================================
print(f"\n{'='*70}")
print(f"  SUMMARY: XLA Loop Indirection vs Predicated")
print(f"  Key: 0% eviction should be ~1.00x (not 0.33x)")
print(f"{'='*70}")
print(f"{'seq':>6} | {'evict%':>6} | {'K':>4} | {'pred ms':>8} | {'loop ms':>8} | {'Δτ':>8} | {'speedup':>8}")
print("-" * 68)
for (seq, pct), r in sorted(results.items()):
    dt = (r['pred'] - r['loop']) / r['pred'] * 100
    spd = r['pred'] / r['loop'] if r['loop'] > 0 else 0
    print(f"{seq:>6} | {pct:>6} | {r['K']:>4} | {r['pred']:>8.3f} | {r['loop']:>8.3f} | {dt:>7.1f}% | {spd:>7.2f}x")

Platform: TPU v5 lite, JAX 0.10.1
Config: Q=1, heads=4, d=128, block=512
Method: jax.lax.fori_loop + jax.lax.dynamic_slice (native XLA)

=== Correctness check (16K, 50% eviction) ===
  predicated vs loop-indirect max err: 0.000000
  PASSED

  8192 tokens (16 blocks)
  dense: 0.264 ms
  pred_0%: 0.257 ms
  loop_0% (K=16): 0.334 ms
    -> Δτ = -30.2%  (0.77x)
  pred_50%: 0.271 ms
  loop_50% (K=8): 0.260 ms
    -> Δτ = +4.1%  (1.04x)
  pred_75%: 0.284 ms
  loop_75% (K=4): 0.223 ms
    -> Δτ = +21.7%  (1.28x)
  pred_90%: 0.256 ms
  loop_90% (K=2): 0.191 ms
    -> Δτ = +25.6%  (1.34x)
  16384 tokens (32 blocks)
  dense: 0.418 ms
  pred_0%: 0.428 ms
  loop_0% (K=32): 0.450 ms
    -> Δτ = -5.2%  (0.95x)
  pred_50%: 0.427 ms
  loop_50% (K=16): 0.346 ms
    -> Δτ = +19.0%  (1.23x)
  pred_75%: 0.431 ms
  loop_75% (K=8): 0.254 ms
    -> Δτ = +41.1%  (1.70x)
  pred_90%: 0.458 ms
  loop_90% (K=4): 0.233 ms
    -> Δτ = +49.1%  (1.96x)
  32768 tokens (64 blocks)
  dense: 0.446 ms
  pred_0%: 0.439 ms
